In [27]:
import pandas as pd
import requests
import numpy as np

In [44]:
from binance.client import Client
client = Client()  # sin API key para datos públicos
velas = client.get_historical_klines("BTCUSDT", Client.KLINE_INTERVAL_1HOUR,
                                     "1 Jan, 2024", "31 Dec, 2025")

In [45]:
velas_df = pd.DataFrame(velas, columns=["fecha", "open", "high", "low", "close", "volume",
                                       "close_time", "quote_asset_volume", "number_of_trades",
                                       "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"])

In [46]:
velas_df["fecha"] = pd.to_datetime(velas_df["fecha"], unit="ms")
velas_df["close_time"] = pd.to_datetime(velas_df["close_time"], unit="ms")

In [47]:
cols = velas_df.select_dtypes(include=[str]).columns

for col in cols:
    velas_df[col] = pd.to_numeric(velas_df[col])

In [48]:
velas_df.to_parquet("velas_df.parquet", engine="pyarrow", index=False)

In [49]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_parquet("velas_df.parquet")
sub = df[(df["fecha"] >= "2025-01-01") & (df["fecha"] < "2025-03-15")]  # un tramo para que se lea

fig = go.Figure(go.Candlestick(
    x=sub["fecha"],
    open=sub["open"], high=sub["high"], low=sub["low"], close=sub["close"],
    name="BTCUSDT 1h",
))
fig.update_layout(title="BTCUSDT 1h", xaxis_rangeslider_visible=False, height=600)
fig.show()